# Generality (FORCED) — predictive allocation grafted onto RAMP (Colab, **NOT paper-grade**)

φ routes each sample's crafting — **full** on its predicted-binding norm, **floor** on the other — keeping RAMP's max + KL-pairing + GP intact. **STEP-1 smoke first** (runs? FLOPs<1.0? l∞ survive?), report, THEN read. Uses your dataset at `/content/drive/MyDrive/attackdro/data` (no re-download). Pinned `1d901cad`.


## 1 · Setup — our repo @ pin, RAMP @ be4971f, copy graft, patch

In [ ]:
!git clone --quiet https://github.com/anhkiet287/attackdro.git 2>/dev/null || (cd attackdro && git fetch --quiet origin)
%cd attackdro
!git checkout --quiet 1d901cad
!git clone --quiet https://github.com/uiuc-focal-lab/RAMP.git external/RAMP 2>/dev/null || true
!cd external/RAMP && git checkout --quiet be4971f04cf8e70bd8255874a1ed2ab489cae682
!cp scripts/ramp/pred_alloc.py external/RAMP/pred_alloc.py
!python scripts/dev/make_ramp_predalloc.py
!pip install --quiet robustbench autoattack timm 2>/dev/null || true
!python -c "import py_compile; py_compile.compile('external/RAMP/RAMP_predalloc.py'); print('compiles')"

## 2 · Drive + data (USE EXISTING dataset, no download)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, torch
DATA='/content/drive/MyDrive/attackdro/data'
EXPDIR='/content/drive/MyDrive/attackdro/results/exploration'; os.makedirs(EXPDIR, exist_ok=True)
assert os.path.isdir(f'{DATA}/cifar-10-batches-py'), f'CIFAR-10 not at {DATA}/cifar-10-batches-py — fix DATA / unpack there'
print('dataset OK:', os.listdir(f'{DATA}/cifar-10-batches-py')[:3])
os.environ['WANDB_MODE']='disabled'
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARN)')

## 3 · STEP 1 — SMOKE (runs? FLOPs<1.0? l∞ survive?)

Read BEFORE Step 2: `attack_flops_ratio < 1.0` (~0.6); `acc_s` (Linf) NOT ~0; no crash.

In [ ]:
!cd external/RAMP && PREDALLOC_FLOOR=2 python -u RAMP_predalloc.py --lr-max 0.05 --lr-schedule=static --at_iter 10 --save_freq 10 --kl --max --gp --lbd 5 --seed 0 --final_eval --data_dir /content/drive/MyDrive/attackdro/data --epochs 3 --eval_freq 3 --fname smoke_predalloc 2>&1 | tee {EXPDIR}/ramp_predalloc_smoke.log | grep -E 'predalloc|acc_s|acc_t|Traceback|Error' | tail -30

### ⛔ STEP-1 GATE — report the smoke to Kiet before continuing
Crashed / ratio not <1.0 / `acc_s`≈0 (l∞ destroyed) → **STOP, report.** Do NOT run Step 2.

## 4 · STEP 2 — short read (ep25): Arm A RAMP-full vs Arm B RAMP+predalloc

In [ ]:
# Arm A = RAMP-full (FLOPs 1.0)
!cd external/RAMP && python -u RAMP.py --lr-max 0.05 --lr-schedule=static --at_iter 10 --save_freq 10 --kl --max --gp --lbd 5 --seed 0 --final_eval --data_dir /content/drive/MyDrive/attackdro/data --epochs 25 --eval_freq 25 --fname armA_rampfull 2>&1 | tee {EXPDIR}/ramp_armA_full.log | grep -E 'acc_s|acc_t|robust|union|clean' | tail -20

In [ ]:
# Arm B = RAMP + predictive allocation
!cd external/RAMP && PREDALLOC_FLOOR=2 python -u RAMP_predalloc.py --lr-max 0.05 --lr-schedule=static --at_iter 10 --save_freq 10 --kl --max --gp --lbd 5 --seed 0 --final_eval --data_dir /content/drive/MyDrive/attackdro/data --epochs 25 --eval_freq 25 --fname armB_predalloc 2>&1 | tee {EXPDIR}/ramp_armB_predalloc.log | grep -E 'predalloc|acc_s|acc_t|robust|union|clean' | tail -25

## 5 · Read → append to generality_predictive_on_ramp.md

In [ ]:
import re, os
b = open(f'{EXPDIR}/ramp_armB_predalloc.log').read() if os.path.exists(f'{EXPDIR}/ramp_armB_predalloc.log') else ''
flops = re.findall(r'attack_flops_ratio = ([0-9.]+)', b); fl = flops[-1] if flops else '?'
lines = ['', '## STEP-2 short read (ep25, 1 seed, RAMP --final_eval)', '',
  f'- Arm B measured attack-FLOPs ratio (last epoch) = **{fl}** (Arm A = 1.0)',
  f'- union / per-norm: {EXPDIR}/ramp_armA_full.log (A) vs ramp_armB_predalloc.log (B)',
  '- SIGNAL: union preserved at FLOPs<1.0 → transfers; drops / l∞ collapses → does not (bounds the claim).',
  '- ep25 pre-drop, 1 seed = SIGNAL not verdict. Positive → proper 5070ti Phase-2 run LATER.']
open(f'{EXPDIR}/generality_predictive_on_ramp.md','a').write('\n'.join(lines)+'\n'); print('\n'.join(lines))

---
**STOP.** Signal, not verdict. Send smoke + read (FLOPs, union, l∞, l1: A vs B) to Kiet.